# Building a DECAT Dataset

This notebook is a helper tool for constructing an `ObsTable` compatible data set from the DECAT visit data in [https://github.com/gnarayan/decat_pointings](https://github.com/gnarayan/decat_pointings). It is based on the [DECAT_qcinv_summary](https://github.com/gnarayan/decat_pointings/blob/main/notebooks/DECAT_qcinv_summary.ipynb) notebook by Melissa Graham.

To run this notebook, users need to first `git clone` the data repository locally and set the `DATA_DIR` to the corresponding local directory.


## Locating the qcinv files

The DECAT data is stored in a nested series of directories.

In [ ]:
import glob
import numpy as np

# The default is to assume the lightcurvelynx and decat_pointings diectories
# are in the same parent directory. You may need to change this parameter if
# your directory structure is different.
DATA_DIR = "../../decat_pointings"

# Get the list of all .qcinv files in the data directory sorted by path.
filenames = sorted(glob.glob(f"{DATA_DIR}/*/*/*.qcinv"))
print(f"Found {len(filenames)} .qcinv files.")

To simplify processing, we add a helper function that will compute the MJD from the file's path and the ut column of the data (which is in the form "HH:MM").

In [ ]:
from astropy.time import Time
from pathlib import Path


def mjd_from_filename(file_path, ut=None):
    """Extract the base MJD of the pointing from the file's path.

    Parameters
    ----------
    file_path : str
        The path to the .qcinv file in the form:
        <PREFIX>/<YYYYMMDD>.qcinv
    ut : str, optional
        The UT time to use for the MJD calculation in the format "HH:MM:SS".
        If None, the time will be set to 00:00:00.
    """
    if ut is None or ut.strip() == "":
        ut = "00:00:00.00"
    elif len(ut.split(":")) == 2:
        ut += ":00.00"
    if len(ut.split(":")) != 3:
        raise ValueError(f"UT time must be in the format 'HH:MM:SS' or 'HH:MM', got '{ut}'")

    file_name = Path(file_path).stem  # Get the file name without extension
    # Check that the file name has the expected format of YYYYMMDD
    if len(file_name) < 8 or not file_name[:8].isdigit():
        raise ValueError(f"File name '{file_name}' does not start with a valid date in the format YYYYMMDD")

    isot_str = f"{file_name[0:4]}-{file_name[4:6]}-{file_name[6:8]}T{ut}"
    mjd = Time(isot_str, format="isot", scale="utc").mjd
    return mjd

Next we build a helper function to parse a single qcinv file. The data is in fixed ASCII format, so we need to read each line and extract the columns manually (although maybe there is a good tool for this):

  * **expid** (characters 0-5) - An experiment ID [We do not use this]
  * **ra** (characters 6-11) - Right Ascension in degrees
  * **dec** (characters 12-17) - Declination in degrees
  * **ut** (characters 19-23) - ut time as a string "HH:MM"
  * **fil** (characters 24-27) - filter name as a string
  * **time** (characters 28-31) - Exposure time in seconds
  * **secz** (characters 32-36) - Airmass
  * **psf** (characters 37-42) - Seeing in arcsec (blank for no data)
  * **sky** (characters 43-48) - sky background expressed in magnitudes from a dark sky* (blank for no data)
  * **cloud** (characters 49-54) - measure of cloudiness in magnitudes* (blank for no data)
  * **teff** (characters 55-60) - effective exposure time* (blank for no data) - Ratio of exposure time where 1.0 indicate that observations have the depth that is expected for typical good observing conditions with the given exposure time.
  * **Object** - An object label [We do not use this]

*Definitions from [NOIR Lab's DECam Users Guide](https://noirlab.edu/science/programs/ctio/instruments/Dark-Energy-Camera/User-Guide/During-Night#GODB). See that page for more details.

In [ ]:
import pandas as pd


def read_single_qcinv_file(file_path, skip_no_noise=False):
    """Read a single qcinv file into a pandas DataFrame.

    This parser is robust to variable-width spacing between columns. It uses
    the first header line (which starts with '#') to determine column names,
    then tokenizes each data row by whitespace.

    Parameters
    ----------
    file_path : str
        The path to the .qcinv file to read.
    skip_no_noise : bool, optional
        If True, rows with no noise information (i.e. blank psf, sky, cloud, teff) will be skipped.
        Default is False, which will include these rows with NaN values for the missing noise information.
    """

    def _safe_float(value):
        try:
            return float(value)
        except (TypeError, ValueError):
            return np.nan

    values = {
        "time": [],
        "ra": [],
        "dec": [],
        "exptime": [],
        "filter": [],
        "secz": [],
        "psf": [],
        "sky": [],
        "cloud": [],
        "teff": [],
    }

    with open(file_path, "r") as f:
        lines = [line.rstrip("\n") for line in f]

        # Read in the data from the file one line at a time, skipping empty lines
        # and comment lines (after the header, which starts with a '#').
        for line in lines:
            stripped = line.strip()
            if not stripped:  # Empty line
                continue
            if stripped.startswith("#"):  # Comment (including header).
                continue
            if stripped.startswith("MJD") or stripped.startswith("date") or stripped.startswith("time"):
                # The last line has a summary of the pointing with "MJD" or "date" or "time" in it.
                # We can skip this line since we already have the UT time from the data rows.
                continue

            tokens = stripped.split()
            if len(tokens) < 7:
                # If there are fewer than 7 tokens, then there is not enough information to create a valid row.
                # We will skip this row. This happens in one file where there seems to be a noise value
                # at the end.
                continue
            if len(tokens) < 10 and skip_no_noise:
                # If there are fewer than 10 tokens, then there is no noise information (psf, sky, cloud, teff).
                # If skip_no_noise is True, we will skip this row. Otherwise, we will include it with NaN values
                # for the missing noise information.
                continue

            values["time"].append(mjd_from_filename(file_path, tokens[3]))  # UT is the 4th column (index 3)
            values["ra"].append(_safe_float(tokens[1]))  # RA is the 2nd column (index 1)
            values["dec"].append(_safe_float(tokens[2]))  # Dec is the 3rd column (index 2)
            values["filter"].append(tokens[4])  # Filter is the 5th column (index 4)
            values["exptime"].append(_safe_float(tokens[5]))  # exptime is the 6th column (index 5)
            values["secz"].append(_safe_float(tokens[6]))  # secz is the 7th column (index 6)

            if len(tokens) > 10:
                values["psf"].append(_safe_float(tokens[7]))  # psf is the 8th column (index 7)
                values["sky"].append(_safe_float(tokens[8]))  # sky is the 9th column (index 8)
                values["cloud"].append(_safe_float(tokens[9]))  # cloud is the 10th column (index 9)
                values["teff"].append(_safe_float(tokens[10]))  # teff is the 11th column (index 10)
            else:
                values["psf"].append(np.nan)
                values["sky"].append(np.nan)
                values["cloud"].append(np.nan)
                values["teff"].append(np.nan)

    return pd.DataFrame(values)

In [ ]:
file0 = "../../decat_pointings/2021A/20210219/20210219.qcinv"
file1 = "../../decat_pointings/2021A/20210524/20210524.qcinv"

data0 = read_single_qcinv_file(file0)
data1 = read_single_qcinv_file(file1)

print("data0 columns:", data0.columns.tolist())
print(data0.head())

print("\ndata1 columns:", data1.columns.tolist())
print(data1.head())

In [ ]:
data1.head()

We can use our helper functions to read in all the files and combine them.

In [ ]:
from tqdm import tqdm

data_frames = []
for file_path in tqdm(filenames, desc="Reading files", unit="file"):
    try:
        df = read_single_qcinv_file(file_path)
        if np.any(df["ra"].isna()) or np.any(df["dec"].isna()) or np.any(df["time"].isna()):
            print(f"Warning: Missing RA, Dec, or time in {file_path}. This file will be skipped.")
            continue

        data_frames.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

full_data = pd.concat(data_frames, ignore_index=True)
print(f"Combined dataset has {len(full_data)} rows.")
print(full_data.head())

Drop any rows with invalid RA, DEC, or time.

In [ ]:
valid_mask = full_data["ra"].notna() & full_data["dec"].notna() & full_data["time"].notna()
print(f"Number of valid pointings: {valid_mask.sum()} out of {len(full_data)} total.")

full_data = full_data[valid_mask].reset_index(drop=True)
print(full_data.head())

We try putting the DECAT data into a DECam data set to test compatibility.

In [ ]:
from lightcurvelynx.obstable.deccam_obstable import DECamObsTable

obs_table = DECamObsTable.from_decat_survey(full_data)

In [ ]:
outfile = "../data/DECAT_dataset.parquet"
full_data.to_parquet(outfile, index=False)
print(f"Saved full dataset to {outfile}")